In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# -------------------------------------------------
# Load the dataset

In [ ]:
# -------------------------------------------------
filename = 'college_placement_dataset_500.csv'

try:
    df = pd.read_csv(filename)
    print("file loaded, shape:", df.shape)
except FileNotFoundError:
    print("Error: could not find", filename, "- make sure it's in the same folder")
    raise

print(df.head())
df.info()

file loaded, shape: (500, 12)
  Student_ID  Gender  Age  ... Aptitude_Score  Communication_Score  Certifications
0     STU001    Male   22  ...             45                   77               3
1     STU002    Male   21  ...             66                   54               3
2     STU003    Male   21  ...             88                   61               0
3     STU004    Male   22  ...             47                   99               3
4     STU005  Female   22  ...             89                   58               0

[5 rows x 12 columns]
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Student_ID           500 non-null    str    
 1   Gender               500 non-null    str    
 2   Age                  500 non-null    int64  
 3   Department           500 non-null    str    
 4   CGPA                 500 non-null    float64
 5   

In [ ]:
# -------------------------------------------------
# Basic cleaning

In [ ]:
# -------------------------------------------------
print("\nmissing values:\n", df.isnull().sum())

dup = df.duplicated().sum()
print("duplicate rows:", dup)
if dup > 0:
    df = df.drop_duplicates()
    print("removed duplicates, new shape:", df.shape)

# check CGPA for outliers using IQR rule
q1 = df['CGPA'].quantile(0.25)
q3 = df['CGPA'].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
cgpa_outliers = df[(df['CGPA'] < low) | (df['CGPA'] > high)]
print(f"\nCGPA Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"outlier range is below {low:.2f} or above {high:.2f}")
print("number of outliers found:", len(cgpa_outliers))

# min-max normalize a few columns (just to show the technique)
scaler = MinMaxScaler()
cols_to_scale = ['CGPA', 'Aptitude_Score', 'Technical_Skills', 'Communication_Score']
scaled = scaler.fit_transform(df[cols_to_scale])
scaled_df = pd.DataFrame(scaled, columns=[c + '_scaled' for c in cols_to_scale])
print("\nnormalized sample:\n", scaled_df.head())


missing values:
 Student_ID             0
Gender                 0
Age                    0
Department             0
CGPA                   0
Backlogs               0
Projects               0
Internship             0
Technical_Skills       0
Aptitude_Score         0
Communication_Score    0
Certifications         0
dtype: int64
duplicate rows: 0

CGPA Q1=6.67, Q3=8.83, IQR=2.16
outlier range is below 3.43 or above 12.08
number of outliers found: 0

normalized sample:
    CGPA_scaled  ...  Communication_Score_scaled
0     0.637813  ...                    0.616667
1     0.029613  ...                    0.233333
2     0.587699  ...                    0.350000
3     0.091116  ...                    0.983333
4     0.077449  ...                    0.300000

[5 rows x 4 columns]


In [ ]:
# -------------------------------------------------
# Build a placement score since the dataset has no
# actual placement column - we made our own rule based
# on what usually matters for campus placements

In [ ]:
# -------------------------------------------------
intern_yes = df['Internship'].apply(lambda x: 1 if x == 'Yes' else 0)

score = (df['CGPA'] / 10 * 35) + (df['Aptitude_Score'] / 100 * 25) + \
        (df['Technical_Skills'] / 10 * 20) + (df['Communication_Score'] / 100 * 15) + \
        (intern_yes * 5)
df['Composite_Score'] = score.round(2)

eligible = (df['CGPA'] >= 6.0) & (df['Backlogs'] <= 1)
df['Placement_Status'] = np.where(eligible & (df['Composite_Score'] >= 60), 'Placed', 'Not Placed')

# rough salary estimate just for placed students
df['Salary_LPA'] = np.where(
    df['Placement_Status'] == 'Placed',
    (3.5 + (df['CGPA'] - 6) * 1.2 + (df['Technical_Skills'] / 10) * 2.5 + df['Projects'] * 0.4).round(2),
    0
)

print("\nplacement status counts:")
print(df['Placement_Status'].value_counts())
print(df['Placement_Status'].value_counts(normalize=True).round(4) * 100)


placement status counts:
Placement_Status
Placed        338
Not Placed    162
Name: count, dtype: int64
Placement_Status
Placed        67.6
Not Placed    32.4
Name: proportion, dtype: float64


In [ ]:
# -------------------------------------------------
# Descriptive stats + hypothesis test on CGPA

In [ ]:
# -------------------------------------------------
print("\nCGPA skew:", round(stats.skew(df['CGPA']), 4))
print("CGPA kurtosis:", round(stats.kurtosis(df['CGPA']), 4))

# checking if avg cgpa is really above 7.5 or if that's just random
t_stat, p_val = stats.ttest_1samp(df['CGPA'], popmean=7.5)
print(f"\nt-test vs 7.5 -> t = {t_stat:.4f}, p = {p_val:.6f}")
if p_val < 0.05:
    print("p < 0.05 so we reject H0, mean CGPA is significantly different from 7.5")
else:
    print("p >= 0.05, not enough evidence to say it's different from 7.5")

# department wise placement %
dept_table = df.groupby('Department')['Placement_Status'].value_counts(normalize=True).unstack().fillna(0) * 100
print("\nplacement % by department:\n", dept_table.round(2))


CGPA skew: -0.0069
CGPA kurtosis: -1.185

t-test vs 7.5 -> t = 4.4206, p = 0.000012
p < 0.05 so we reject H0, mean CGPA is significantly different from 7.5

placement % by department:
 Placement_Status  Not Placed  Placed
Department                          
Civil                  40.26   59.74
Computer               31.76   68.24
Electrical             32.14   67.86
Electronics            30.00   70.00
IT                     35.37   64.63
Mechanical             25.61   74.39


In [ ]:
# -------------------------------------------------
# Graphs

In [ ]:
# -------------------------------------------------
plt.figure(figsize=(8, 5))
sns.histplot(df['CGPA'], kde=True, color='#2b5c8f', bins=20)
plt.title('Distribution of CGPA')
plt.xlabel('CGPA')
plt.tight_layout()
plt.savefig('fig1_cgpa_distribution.png')
plt.close()

plt.figure(figsize=(7, 5))
sns.boxplot(x='Placement_Status', y='CGPA', data=df, palette='Set2')
plt.title('CGPA vs Placement Status')
plt.tight_layout()
plt.savefig('fig2_cgpa_placement_box.png')
plt.close()

plt.figure(figsize=(10, 5))
sns.countplot(x='Department', hue='Placement_Status', data=df, palette='viridis')
plt.title('Placement by Department')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig('fig3_dept_placement_status.png')
plt.close()

plt.figure(figsize=(9, 7))
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('fig4_correlation_heatmap.png')
plt.close()

plt.figure(figsize=(8, 6))
sns.scatterplot(x='Aptitude_Score', y='Communication_Score', hue='Placement_Status', data=df, alpha=0.7)
plt.title('Aptitude vs Communication')
plt.tight_layout()
plt.savefig('fig5_aptitude_vs_communication.png')
plt.close()

plt.figure(figsize=(7, 5))
sns.countplot(x='Internship', hue='Placement_Status', data=df, palette='pastel')
plt.title('Internship vs Placement')
plt.tight_layout()
plt.savefig('fig6_internship_impact.png')
plt.close()

print("\nsaved 6 plots")


saved 6 plots


In [ ]:
# -------------------------------------------------
# Random Forest model

In [ ]:
# -------------------------------------------------
feature_cols = ['CGPA', 'Backlogs', 'Projects', 'Technical_Skills',
                 'Aptitude_Score', 'Communication_Score', 'Certifications']
X = df[feature_cols]
y = df['Placement_Status'].apply(lambda x: 1 if x == 'Placed' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\ntrain size: {len(X_train)}, test size: {len(X_test)}")

model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"\naccuracy: {acc*100:.2f}%")
print(classification_report(y_test, pred, target_names=['Not Placed', 'Placed']))
print("confusion matrix:\n", confusion_matrix(y_test, pred))

imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nfeature importance:\n", imp.round(4))

print("\ndone")


train size: 400, test size: 100

accuracy: 97.00%
              precision    recall  f1-score   support

  Not Placed       1.00      0.91      0.95        32
      Placed       0.96      1.00      0.98        68

    accuracy                           0.97       100
   macro avg       0.98      0.95      0.96       100
weighted avg       0.97      0.97      0.97       100

confusion matrix:
 [[29  3]
 [ 0 68]]

feature importance:
 CGPA                   0.3905
Backlogs               0.2682
Technical_Skills       0.1036
Aptitude_Score         0.1033
Communication_Score    0.0760
Projects               0.0347
Certifications         0.0237
dtype: float64

done
